In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import rand, round
import string
import random

spark = SparkSession.builder.getOrCreate()

# Parameters
num_rows = 1_000_000  # 1 million rows
names = ["John", "Ana", "Ravi", "Maya", "Leo", "Sara", "Tom", "Nina"]

# Generate large DataFrame
df_large = spark.range(num_rows) \
    .withColumnRenamed("id", "id") \
    .withColumn("name", round(rand()*len(names)).cast("int")) \
    .withColumn("salary", (rand()*1_000_000).cast("int"))

# Replace integer name with random names
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def get_name(idx):
    return names[idx % len(names)]

name_udf = udf(get_name, StringType())
df_large = df_large.withColumn("name", name_udf("name"))

df_large.show(5)

+---+----+------+
| id|name|salary|
+---+----+------+
|  0| Leo|745111|
|  1| Ana|830565|
|  2| Ana|661574|
|  3| Ana|528025|
|  4| Ana|807957|
+---+----+------+
only showing top 5 rows


In [0]:
csv_path = "/Volumes/workspace/default/biswajit/large_employees_csv"
df_large.write.option("header", "true").mode("overwrite").csv(csv_path)

In [0]:
parquet_path = "/Volumes/workspace/default/biswajit/large_employees_parquet"
avro_path = "/Volumes/workspace/default/biswajit/large_employees_avro"

# Parquet
df_large.write.mode("overwrite").parquet(parquet_path)

# Avro
df_large.write.format("avro").mode("overwrite").save(avro_path)

In [0]:
import time

# CSV read time
start = time.time()
spark.read.option("header","true").csv(csv_path).count()
csv_time = time.time() - start

# Parquet read time
start = time.time()
spark.read.parquet(parquet_path).count()
parquet_time = time.time() - start

# Avro read time
start = time.time()
spark.read.format("avro").load(avro_path).count()
avro_time = time.time() - start

print(f"CSV read time: {csv_time:.4f}s")
print(f"Parquet read time: {parquet_time:.4f}s")
print(f"Avro read time: {avro_time:.4f}s")

CSV read time: 2.3557s
Parquet read time: 1.0369s
Avro read time: 1.3104s


In [0]:
from pyspark.sql.functions import length, concat_ws, sum as _sum

def approx_size(df):
    return df.withColumn("row_len", length(concat_ws(",", *df.columns))) \
             .agg(_sum("row_len")).collect()[0][0]

# Read back
df_csv = spark.read.option("header","true").csv(csv_path)
df_parquet = spark.read.parquet(parquet_path)
df_avro = spark.read.format("avro").load(avro_path)

print("Approx sizes (chars):")
print("CSV:", approx_size(df_csv))
print("Parquet:", approx_size(df_parquet))
print("Avro:", approx_size(df_avro))

Approx sizes (chars):
CSV: 17403205
Parquet: 17403205
Avro: 17403205
